## Data Preparation

This notebook prepares the Nymeria eye gaze dataset for analysis. It assumes the raw and
processed data folders have been downloaded via the NymeriaGazeToolkit download script.

**What this notebook does:**

1. **Metadata enrichment** — The base `metadata.csv` from the huggingface dataset does not include
session-level fields needed for analysis. We parse each session's `metadata.json` to extract
whether a session involved two participants, and the recorded action and head-tracking durations.

2. **Noise flagging** — Sessions where the difference between head-tracking duration and
action duration exceeds mean + 3 SD are flagged and excluded. This removes sessions with
abnormally long setup or teardown periods that would corrupt the gaze signal.

3. **Sampling rate detection** — The dataset contains sessions recorded at both 10 Hz and
30 Hz. We detect the sampling rate per session from the gaze timestamps and flag 30 Hz
sessions for exclusion, as our analysis assumes a consistent 10 Hz signal.

4. **Catalog filtering** — We apply four quality filters: 10 Hz only, no noise flag,
has gaze data, and one session per participant per activity (earliest recording kept).
The resulting catalog is saved as `catalog_filtered.csv`.

5. **Session preprocessing** — All filtered sessions are loaded and trimmed using the
per-session trim times derived from the noise computation. Preprocessed sessions are
saved as parquet files to `data/processed/sessions_preprocessed/` so that analysis
notebooks can load them instantly without reprocessing.


In [11]:
from pathlib import Path
import json
import pandas as pd
import nymeria_gaze_tools as ngt

# To download the dataset: run downloadScripts/download_from_hf.py
DATA_ROOT = Path("../data/processed")
RAW_ROOT  = Path("../data/raw")

print(f"metadata.csv exists : {(DATA_ROOT / 'metadata.csv').exists()}")
print(f"Raw data root exists: {RAW_ROOT.exists()}")
print(f"Raw participant dirs : {len(list(RAW_ROOT.iterdir())) if RAW_ROOT.exists() else 'N/A'}")


metadata.csv exists : True
Raw data root exists: True
Raw participant dirs : 237


In [12]:
catalog = pd.read_csv(DATA_ROOT / "metadata.csv")

print(f"Shape: {catalog.shape}  ({catalog['fake_name'].nunique()} unique participants)")
print(f"\nColumns:\n{list(catalog.columns)}")
catalog.head(3)


Shape: (1100, 24)  (236 unique participants)

Columns:
['sequence_uid', 'date', 'session_id', 'fake_name', 'act_id', 'location', 'script', 'participant_gender', 'participant_height_cm', 'participant_weight_kg', 'participant_bmi', 'participant_age_group', 'participant_ethnicity', 'gaze_type', 'has_gaze_data', 'has_two_participants', 'pt2', 'action_duration_sec', 'head_duration_sec', 'noise_sec', 'trim_start_sec', 'trim_end_sec', 'noise_flag', 'sampling_rate_hz']


,sequence_uid,date,session_id,fake_name,act_id,location,script,participant_gender,participant_height_cm,participant_weight_kg,...,has_gaze_data,has_two_participants,pt2,action_duration_sec,head_duration_sec,noise_sec,trim_start_sec,trim_end_sec,noise_flag,sampling_rate_hz
0,20230628_s0_adriana_gonzalez_act0_e587mn,20230628,s0,adriana_gonzalez,act0,Loc_10,S11-Laundary,Male,177.0,81.0,...,True,False,NaN,968.012,1198.474971,230.462971,115.231486,115.231486,False,10
1,20230628_s0_adriana_gonzalez_act1_2vshuw,20230628,s0,adriana_gonzalez,act1,Loc_10,S1-Relax_at_home,Male,177.0,81.0,...,True,False,NaN,916.453,1033.734671,117.281671,58.640835,58.640835,False,10
2,20230628_s0_adriana_gonzalez_act4_rzxlmo,20230628,s0,adriana_gonzalez,act4,Loc_10,S7-Cooking,Male,177.0,81.0,...,True,False,NaN,932.051,1046.799244,114.748244,57.374122,57.374122,False,10


In [13]:
def load_raw_json_fields(raw_root: Path) -> pd.DataFrame:
    records, missing = [], []
    for json_path in sorted(raw_root.glob("*/*/metadata.json")):
        try:
            with open(json_path) as f:
                m = json.load(f)
            uid = f"{m['date']}_{m['session_id']}_{m['fake_name']}_{m['act_id']}_{m['uid']}"
            records.append({
                "sequence_uid":        uid,
                "has_two_participants": m.get("has_two_participants"),
                "pt2":                 m.get("pt2"),
                "action_duration_sec": m.get("action_duration_sec"),
                "head_duration_sec":   m.get("head_duration_sec"),
            })
        except Exception as e:
            missing.append((str(json_path), str(e)))
    df = pd.DataFrame(records)
    print(f"JSONs parsed: {len(df)}  |  Errors: {len(missing)}")
    return df

json_fields = load_raw_json_fields(RAW_ROOT)

# Only merge columns not already present in catalog (avoids _x/_y conflicts)
new_cols = ["sequence_uid"] + [c for c in json_fields.columns if c not in catalog.columns and c != "sequence_uid"]
catalog_enriched = catalog.merge(json_fields[new_cols], on="sequence_uid", how="left")

print(f"Rows matched: {catalog_enriched['has_two_participants'].notna().sum()} / {len(catalog_enriched)}")
print(f"\nhas_two_participants breakdown:")
print(catalog_enriched["has_two_participants"].value_counts())

JSONs parsed: 1100  |  Errors: 0
Rows matched: 1100 / 1100

has_two_participants breakdown:
has_two_participants
False    883
True     217
Name: count, dtype: int64


In [14]:
# Flag sessions where setup/teardown noise is an outlier (mean + 3 SD threshold)
catalog_enriched["noise_sec"]      = catalog_enriched["head_duration_sec"] - catalog_enriched["action_duration_sec"]
catalog_enriched["trim_start_sec"] = catalog_enriched["noise_sec"] / 2
catalog_enriched["trim_end_sec"]   = catalog_enriched["noise_sec"] / 2

threshold = catalog_enriched["noise_sec"].mean() + 3 * catalog_enriched["noise_sec"].std()
catalog_enriched["noise_flag"] = catalog_enriched["noise_sec"] > threshold

print(f"Noise threshold: {threshold:.1f} sec  (mean + 3 SD)")
print(f"Sessions flagged: {catalog_enriched['noise_flag'].sum()}")
print(catalog_enriched[catalog_enriched["noise_flag"]][["fake_name", "script", "noise_sec"]])


Noise threshold: 454.2 sec  (mean + 3 SD)
Sessions flagged: 3
          fake_name                  script    noise_sec
427  erin_mccormick        S9-Making_a_mess   564.210056
494  heather_becker        S1-Relax_at_home   476.457109
625    julie_taylor  S3-Welcome_to_my_place  2554.724876


In [15]:
# Detect sampling rate from the first 50 rows of each session's timestamp column
def get_sampling_rate(uid, data_root):
    path = data_root / "eye_gaze" / f"{uid}.csv"
    df = pd.read_csv(path, usecols=["tracking_timestamp_us"], nrows=50)
    median_interval_s = df["tracking_timestamp_us"].diff().dropna().median() / 1e6
    return round(1.0 / median_interval_s)

print("Computing sampling rates...")
catalog_enriched["sampling_rate_hz"] = catalog_enriched["sequence_uid"].apply(
    lambda uid: get_sampling_rate(uid, DATA_ROOT)
)
print(catalog_enriched["sampling_rate_hz"].value_counts().to_string())


Computing sampling rates...
sampling_rate_hz
10    1066
30      34


In [16]:
# Save the enriched catalog
catalog_enriched.to_csv(DATA_ROOT / "metadata.csv", index=False)

print(f"Saved: {DATA_ROOT / 'metadata.csv'}")
print(f"Shape: {catalog_enriched.shape}")
print(f"Columns: {list(catalog_enriched.columns)}")


Saved: ../data/processed/metadata.csv
Shape: (1100, 24)
Columns: ['sequence_uid', 'date', 'session_id', 'fake_name', 'act_id', 'location', 'script', 'participant_gender', 'participant_height_cm', 'participant_weight_kg', 'participant_bmi', 'participant_age_group', 'participant_ethnicity', 'gaze_type', 'has_gaze_data', 'has_two_participants', 'pt2', 'action_duration_sec', 'head_duration_sec', 'noise_sec', 'trim_start_sec', 'trim_end_sec', 'noise_flag', 'sampling_rate_hz']


In [17]:
# Keep one session per participant per activity: 10 Hz only, clean trim, has gaze data, earliest recording
catalog_filtered = catalog_enriched.copy()
catalog_filtered = catalog_filtered[catalog_filtered["sampling_rate_hz"] == 10]
catalog_filtered = catalog_filtered[catalog_filtered["noise_flag"] == False]
catalog_filtered = catalog_filtered[catalog_filtered["has_gaze_data"] == True]
catalog_filtered = (catalog_filtered
                    .sort_values("date")
                    .drop_duplicates(["fake_name", "script"])
                    .reset_index(drop=True))

catalog_filtered.to_csv(DATA_ROOT / "catalog_filtered.csv", index=False)

print(f"Raw sessions:      {len(catalog_enriched)}")
print(f"Filtered sessions: {len(catalog_filtered)}")
print(f"Unique participants: {catalog_filtered['fake_name'].nunique()}")
print(f"Unique activities:  {catalog_filtered['script'].nunique()}")
print(f"\nSaved: {DATA_ROOT / 'catalog_filtered.csv'}")


Raw sessions:      1100
Filtered sessions: 980
Unique participants: 228
Unique activities:  20

Saved: ../data/processed/catalog_filtered.csv


In [18]:
# Load and preprocess every session — trim the setup/teardown noise from each recording
session_dfs = {}
skipped = []

print(f"Loading {len(catalog_filtered)} sessions...")
for i, (_, row) in enumerate(catalog_filtered.iterrows()):
    if i % 50 == 0:
        print(f"  {i}/{len(catalog_filtered)}")
    try:
        raw = ngt.load_session(row["sequence_uid"], data_root=DATA_ROOT)
        session_dfs[row["sequence_uid"]] = ngt.preprocess(
            raw,
            trim_start_min=row["trim_start_sec"] / 60,
            trim_end_min=row["trim_end_sec"] / 60,
        )
    except Exception as e:
        skipped.append((row["sequence_uid"], row["script"], str(e)))

print(f"\nDone. Loaded: {len(session_dfs)}  |  Skipped: {len(skipped)}")
for uid, script, err in skipped:
    print(f"  SKIP {script} | {uid} | {err}")


Loading 980 sessions...
  0/980
  50/980
  100/980
  150/980
  200/980
  250/980
  300/980
  350/980
  400/980
  450/980
  500/980
  550/980
  600/980
  650/980
  700/980
  750/980
  800/980
  850/980
  900/980
  950/980

Done. Loaded: 979  |  Skipped: 1
  SKIP S6-Dance | 20230726_s1_thomas_nixon_act2_bsfnuw | 'left_yaw_rads_cpf'


In [19]:
# Save each preprocessed session to disk so analysis notebooks load instantly
SESSIONS_DIR = DATA_ROOT / "sessions_preprocessed"
SESSIONS_DIR.mkdir(exist_ok=True)

for uid, df in session_dfs.items():
    df.to_parquet(SESSIONS_DIR / f"{uid}.parquet", index=False)

print(f"Saved {len(session_dfs)} sessions to {SESSIONS_DIR}")


Saved 979 sessions to ../data/processed/sessions_preprocessed
